# **Data Cleaning**

## Objectives

The objectives of this notebook are to:

- Load the merged dataset created during the data collection stage.
- Assess the quality of the dataset by checking for missing values, duplicate records and inconsistent data types.
- Clean and standardise the dataset where necessary.
- Save the cleaned dataset for exploratory data analysis and machine learning.


## Inputs

- `data/processed/all_players.csv` – The merged Premier League player statistics dataset created in the Data Collection notebook.

## Outputs

- `data/processed/cleaned_players.csv` – The cleaned and validated dataset, ready for exploratory data analysis and machine learning.

## Additional Comments

This notebook focuses on data quality rather than data analysis. Cleaning tasks are performed to improve the reliability and consistency of the dataset before exploratory data analysis and model development. Any assumptions or cleaning decisions made during this stage will be documented throughout the notebook.

---

### Change working directory

In [2]:
import os
current_dir = os.getcwd()
current_dir

'c:\\code\\premier-league-predictor\\premier-league-predictor\\jupyter_notebooks'

In [3]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


In [4]:
current_dir = os.getcwd()
current_dir

'c:\\code\\premier-league-predictor\\premier-league-predictor'

### Import packages

All data loading and inspection in this notebook uses `pandas`. No additional libraries are needed at this stage.

In [5]:

import pandas as pd

## Load processed dataset

The merged dataset created during the data collection stage is loaded from the `data/processed` directory. This dataset will be assessed for data quality issues before exploratory data analysis.

In [6]:
data_path = "data/processed"
file_path = os.path.join(data_path, "all_players.csv")
df = pd.read_csv(file_path)

## Inspect dataset

The dataset is inspected to confirm it has loaded correctly and to obtain an initial understanding of its size, structure and data types before cleaning begins.

In [7]:
df.shape


(8206, 53)

In [8]:
df.head()

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Shooting accuracy %,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks
0,Rolando Aarons,Midfielder,10,NaN,NaN,13.0,77%,NaN,0.0,6.0,...,50%,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Almen Abdi,Midfielder,32,NaN,NaN,83.0,78%,NaN,10.0,32.0,...,26%,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Abdul Rahman Baba,Defender,15,2.0,13.0,47.0,83%,0.0,1.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Mehdi Abeid,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0%,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Tammy Abraham,Forward,2,NaN,NaN,0.0,NaN,NaN,1.0,0.0,...,0%,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8206 entries, 0 to 8205
Data columns (total 53 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    8206 non-null   object 
 1   Position                8206 non-null   object 
 2   Appearances             8206 non-null   int64  
 3   Clean sheets            3579 non-null   float64
 4   Goals conceded          3579 non-null   float64
 5   Tackles                 7244 non-null   float64
 6   Tackle success %        5531 non-null   object 
 7   Last man tackles        2617 non-null   float64
 8   Blocked shots           7244 non-null   float64
 9   Interceptions           7244 non-null   float64
 10  Clearances              7244 non-null   float64
 11  Headed Clearance        7244 non-null   float64
 12  Clearances off line     2617 non-null   float64
 13  Recoveries              5531 non-null   float64
 14  Duels won               5531 non-null   

## Investigate missing values

Missing values are assessed to determine whether they represent genuine missing data or values that are not applicable for certain player positions. Understanding the reason for missing values is essential before deciding how they should be handled.

In [10]:
df.isnull().sum().sort_values(ascending=False)

Catches                   7244
High Claims               7244
Punches                   7244
Penalties saved           7244
Saves                     7244
Goal Kicks                7244
Throw outs                7244
Sweeper clearances        7244
Last man tackles          5589
Clearances off line       5589
Own goals                 4627
Clean sheets              4627
Goals conceded            4627
Penalties scored          3579
Shots                     3579
Shots on target           3579
Big chances missed        3579
Goals per match           3579
Freekicks scored          3579
Shooting accuracy %       3579
Aerial battles lost       2675
Cross accuracy %          2675
Successful 50/50s         2675
Duels lost                2675
Duels won                 2675
Tackle success %          2675
Recoveries                2675
Aerial battles won        2675
Through balls             2675
Errors leading to goal    1713
Accurate long balls       1713
Offsides                   962
Blocked 

### Investigate goalkeeper-specific statistics

Several goalkeeper-specific features contain a large number of missing values. This investigation determines whether these missing values represent poor data quality or statistics that are only applicable to goalkeepers.

In [11]:
df["Position"].value_counts()

Position
Midfielder    2884
Defender      2635
Forward       1725
Goalkeeper     962
Name: count, dtype: int64

In [12]:
goalkeepers = (df["Position"] == "Goalkeeper").sum()

In [13]:
outfield_players = len(df) - goalkeepers

In [14]:
missing_saves = df["Saves"].isna().sum()

In [15]:
print(f"Goalkeepers: {goalkeepers}")
print(f"Outfield players: {outfield_players}")
print(f"Missing 'Saves' values: {missing_saves}")

Goalkeepers: 962
Outfield players: 7244
Missing 'Saves' values: 7244


In [16]:
goalkeeper_columns = [
    "Saves",
    "Penalties saved",
    "Punches",
    "High Claims",
    "Catches",
    "Sweeper clearances",
    "Throw outs",
    "Goal Kicks",
]

df[goalkeeper_columns].isna().sum()

Saves                 7244
Penalties saved       7244
Punches               7244
High Claims           7244
Catches               7244
Sweeper clearances    7244
Throw outs            7244
Goal Kicks            7244
dtype: int64

### Findings

The investigation confirmed that all goalkeeper-specific statistics contain exactly **7,244** missing values, matching the number of outfield players in the dataset. This demonstrates that these values are not missing due to incomplete data collection; they are not applicable to outfield players. These columns will therefore be handled differently from genuinely missing data during the cleaning process.

### Investigate defensive statistics

Two defensive features, `Last man tackles` and `Clearances off line`, contain a large number of missing values. This investigation determines whether these missing values are associated with particular playing positions or represent incomplete data.

In [17]:
defenders = (df["Position"] == "Defender").sum()

In [18]:
other_players = len(df) - defenders

In [19]:
missing_last_man_tackles = df["Last man tackles"].isna().sum()

In [20]:
print(f"Defenders: {defenders}")
print(f"Other players: {other_players}")
print(f"Missing 'Last man tackles' values: {missing_last_man_tackles}")

Defenders: 2635
Other players: 5571
Missing 'Last man tackles' values: 5589


In [21]:
df.loc[df["Last man tackles"].notna(), "Position"].value_counts()

Position
Defender      2609
Midfielder       8
Name: count, dtype: int64

### Findings

The investigation showed that `Last man tackles` is primarily recorded for defenders, with only a small number of midfielders having recorded values. This indicates that the missing values are expected for most attacking players rather than representing missing or incomplete data.

### Investigate attacking statistics

Several attacking statistics also contain a large number of missing values. This investigation determines whether these missing values represent statistics that are only applicable to certain players or whether they indicate missing or incomplete data.

In [22]:
df.loc[df["Shots"].notna(), "Position"].value_counts()

Position
Midfielder    2876
Forward       1725
Defender        26
Name: count, dtype: int64

In [23]:
for position in df["Position"].unique():
    missing = df.loc[df["Position"] == position, "Shots"].isna().sum()
    print(position, missing)

Midfielder 8
Defender 2609
Forward 0
Goalkeeper 962


In [24]:
df.loc[
    (df["Position"] == "Defender") & (df["Shots"].isna()),
    ["Name", "Appearances", "Goals", "Shots"]
].head(20)

,Name,Appearances,Goals,Shots
2,Abdul Rahman Baba,15,0,NaN
12,Nathan Aké,24,1,NaN
14,Alberto Moreno,32,1,NaN
16,Toby Alderweireld,38,4,NaN
18,Trent Alexander-Arnold,0,0,NaN
25,Daniel Amartey,5,0,NaN
26,Jordan Amavi,10,0,NaN
31,Angeliño,0,0,NaN
32,Gabriele Angella,0,0,NaN
45,César Azpilicueta,37,2,NaN


In [25]:
df.loc[
    (df["Goals"] > 0) & (df["Shots"].isna()),
    ["Name", "Position", "Goals", "Shots"]
]

,Name,Position,Goals,Shots
12,Nathan Aké,Defender,1,NaN
14,Alberto Moreno,Defender,1,NaN
16,Toby Alderweireld,Defender,4,NaN
45,César Azpilicueta,Defender,2,NaN
48,Leighton Baines,Defender,2,NaN
...,...,...,...,...
8168,Ben White,Defender,4,NaN
8192,Illia Zabarnyi,Defender,1,NaN
8196,Zanka,Defender,1,NaN
8201,Oleksandr Zinchenko,Defender,1,NaN


### Findings

Unlike the goalkeeper-specific statistics, the missing values in attacking statistics do not appear to be structural. The investigation identified **635 player records** with at least one recorded goal but missing shot statistics. Since a player cannot score without taking a shot, this indicates that these missing values represent incomplete source data rather than statistics that are not applicable.

***

## Decide how to handle missing values

The previous investigations identified different types of missing values within the dataset. Rather than applying a single approach to all missing data, each group of features will be handled according to the reason the values are missing.

### Goalkeeper-specific statistics

The investigation confirmed that missing values in goalkeeper-specific statistics are structural rather than the result of incomplete data. As these statistics are not applicable to outfield players, the missing values are replaced with `0` to indicate that no value was recorded for those statistics.

In [26]:
goalkeeper_columns = [
    "Saves",
    "Penalties saved",
    "Punches",
    "High Claims",
    "Catches",
    "Sweeper clearances",
    "Throw outs",
    "Goal Kicks",
]

In [27]:
df[goalkeeper_columns] = df[goalkeeper_columns].fillna(0)
df[goalkeeper_columns].isna().sum()

Saves                 0
Penalties saved       0
Punches               0
High Claims           0
Catches               0
Sweeper clearances    0
Throw outs            0
Goal Kicks            0
dtype: int64

### Outcome

The goalkeeper-specific features no longer contain missing values. These `NaN` values were replaced with `0` because the statistics are not applicable to outfield players rather than representing incomplete data.

In [28]:
df.columns.tolist()

['Name',
 'Position',
 'Appearances',
 'Clean sheets',
 'Goals conceded',
 'Tackles',
 'Tackle success %',
 'Last man tackles',
 'Blocked shots',
 'Interceptions',
 'Clearances',
 'Headed Clearance',
 'Clearances off line',
 'Recoveries',
 'Duels won',
 'Duels lost',
 'Successful 50/50s',
 'Aerial battles won',
 'Aerial battles lost',
 'Own goals',
 'Errors leading to goal',
 'Assists',
 'Passes',
 'Passes per match',
 'Big chances created',
 'Crosses',
 'Cross accuracy %',
 'Through balls',
 'Accurate long balls',
 'Yellow cards',
 'Red cards',
 'Fouls',
 'Offsides',
 'Goals',
 'Headed goals',
 'Goals with right foot',
 'Goals with left foot',
 'Hit woodwork',
 'Goals per match',
 'Penalties scored',
 'Freekicks scored',
 'Shots',
 'Shots on target',
 'Shooting accuracy %',
 'Big chances missed',
 'Saves',
 'Penalties saved',
 'Punches',
 'High Claims',
 'Catches',
 'Sweeper clearances',
 'Throw outs',
 'Goal Kicks']

In [29]:
attacking_columns = [
    "Goals",
    "Goals per match",
    "Shots",
    "Shots on target",
    "Shooting accuracy %",
    "Big chances missed",
    "Hit woodwork",
    "Assists",
    "Big chances created",
]

In [30]:
df[attacking_columns].isna().sum()

Goals                     0
Goals per match        3579
Shots                  3579
Shots on target        3579
Shooting accuracy %    3579
Big chances missed     3579
Hit woodwork            962
Assists                   0
Big chances created     962
dtype: int64

In [31]:
df.loc[df["Hit woodwork"].notna(), "Position"].value_counts()

Position
Midfielder    2884
Defender      2635
Forward       1725
Name: count, dtype: int64

In [32]:
df.loc[df["Big chances created"].notna(), "Position"].value_counts()

Position
Midfielder    2884
Defender      2635
Forward       1725
Name: count, dtype: int64

## Summary of missing value categories

| Category | Features | Planned action |
|----------|----------|----------------|
| Structural missing values | Saves, Penalties saved, Punches, High Claims, Catches, Sweeper clearances, Throw outs, Goal Kicks, Hit woodwork, Big chances created | Replace `NaN` with `0` |
| Position-specific statistics | Last man tackles, Clearances off line | Investigate further before deciding |
| Incomplete source data | Goals per match, Shots, Shots on target, Shooting accuracy %, Big chances missed | Do not replace with `0`; investigate appropriate handling |

### Update structural missing value features

Further investigation showed that `Hit woodwork` and `Big chances created` are complete for all outfield players and only missing for goalkeepers. These features are therefore added to the list of structural missing values and their missing values are replaced with `0`.

In [33]:
structural_columns = [
    "Saves",
    "Penalties saved",
    "Punches",
    "High Claims",
    "Catches",
    "Sweeper clearances",
    "Throw outs",
    "Goal Kicks",
    "Hit woodwork",
    "Big chances created",
]

df[structural_columns] = df[structural_columns].fillna(0)

In [34]:
df[structural_columns].isna().sum()

Saves                  0
Penalties saved        0
Punches                0
High Claims            0
Catches                0
Sweeper clearances     0
Throw outs             0
Goal Kicks             0
Hit woodwork           0
Big chances created    0
dtype: int64

### Outcome

The structural missing values have now been replaced with `0`. A verification check confirmed that these features no longer contain missing values. This approach was appropriate because the missing values represented statistics that were not applicable to certain player positions rather than incomplete data.

***

## Handle position-specific statistics

The investigation showed that some statistics are primarily associated with specific playing positions rather than being applicable to all players. The following section determines the most appropriate way to handle these remaining missing values.

In [35]:
position_specific_columns = [
    "Last man tackles",
    "Clearances off line",
]

In [36]:
df[position_specific_columns].isna().sum()

Last man tackles       5589
Clearances off line    5589
dtype: int64

### Investigation findings

The investigation showed that `Last man tackles` and `Clearances off line` contain the same number of missing values. These statistics are primarily recorded for defenders, with only a very small number of midfielders having recorded values. The missing values are therefore position-specific rather than random.

***

## Handle incomplete source data

The remaining missing values occur in attacking statistics where the investigation showed that the data is incomplete rather than structurally missing. These values are reviewed separately to determine whether they should be retained, removed or imputed.

In [37]:
incomplete_columns = [
    "Goals per match",
    "Shots",
    "Shots on target",
    "Shooting accuracy %",
    "Big chances missed",
]

In [38]:
df[incomplete_columns].isna().sum()

Goals per match        3579
Shots                  3579
Shots on target        3579
Shooting accuracy %    3579
Big chances missed     3579
dtype: int64

### Investigation findings

The investigation showed that these attacking statistics contain genuine missing values rather than structural missing values. Players with recorded goals were found to have missing shot statistics, demonstrating that the source data is incomplete. Replacing these values with `0` would introduce inaccurate information into the dataset.

In [39]:
df.isna().sum().sort_values(ascending=False)

Last man tackles          5589
Clearances off line       5589
Own goals                 4627
Clean sheets              4627
Goals conceded            4627
Shooting accuracy %       3579
Goals per match           3579
Freekicks scored          3579
Penalties scored          3579
Big chances missed        3579
Shots on target           3579
Shots                     3579
Aerial battles lost       2675
Through balls             2675
Aerial battles won        2675
Recoveries                2675
Tackle success %          2675
Duels won                 2675
Duels lost                2675
Successful 50/50s         2675
Cross accuracy %          2675
Errors leading to goal    1713
Accurate long balls       1713
Headed goals               962
Blocked shots              962
Headed Clearance           962
Interceptions              962
Clearances                 962
Tackles                    962
Goals with left foot       962
Goals with right foot      962
Offsides                   962
Crosses 

In [40]:
df.loc[df["Clean sheets"].notna(), "Position"].value_counts()

Position
Defender      2609
Goalkeeper     962
Midfielder       8
Name: count, dtype: int64

In [41]:
df.loc[df["Goals conceded"].notna(), "Position"].value_counts()

Position
Defender      2609
Goalkeeper     962
Midfielder       8
Name: count, dtype: int64

In [42]:
df.loc[
    (df["Position"] == "Defender") &
    (df["Clean sheets"].isna()),
    ["Name", "Appearances", "Goals", "Clean sheets"]
]

,Name,Appearances,Goals,Clean sheets
463,Sam McQueen,0,0,NaN
533,Paddy McCarthy,0,0,NaN
1025,Dael Fry,0,0,NaN
1280,Sam McQueen,13,0,NaN
1366,Paddy McCarthy,0,0,NaN
1592,Oleksandr Zinchenko,0,0,NaN
2080,Sam McQueen,7,0,NaN
2387,Oleksandr Zinchenko,8,0,NaN
2528,Trevoh Chalobah,0,0,NaN
2718,Kortney Hause,0,0,NaN


#### Further investigation of defensive statistics such as `Clean sheets` identified players with substantial numbers of appearances but missing values. This suggests that these missing values also represent incomplete source data rather than statistics that are not applicable.

### Investigate position-dependent performance statistics

Several performance statistics contain **2,675** missing values. The following investigation determines whether these missing values are position-specific or represent incomplete source data.

In [43]:
df.loc[df["Duels won"].notna(), "Position"].value_counts()

Position
Midfielder    2853
Defender      2635
Forward         43
Name: count, dtype: int64

In [50]:
df.loc[
    (df["Position"] == "Forward") &
    (df["Duels won"].isna()),
    ["Name", "Appearances", "Goals", "Duels won", "Recoveries"]
].head(20)

,Name,Appearances,Goals,Duels won,Recoveries
4,Tammy Abraham,2,0,NaN,NaN
6,Emmanuel Adebayor,12,1,NaN,NaN
9,Benik Afobe,15,4,NaN,NaN
10,Gabriel Agbonlahor,15,1,NaN,NaN
11,Sergio Agüero,30,24,NaN,NaN
13,Chuba Akpom,0,0,NaN,NaN
19,Alexandre Pato,2,1,NaN,NaN
33,Victor Anichebe,10,0,NaN,NaN
38,Adam Armstrong,0,0,NaN,NaN
39,Marko Arnautovic,34,11,NaN,NaN


### Decision

The remaining missing values will be retained as `NaN`. The investigations showed that these values do not represent structural missing values and cannot be safely replaced with `0` without introducing inaccurate information. The remaining missing values represent either incomplete source data or statistics that were not consistently recorded across all player roles. Any further handling of these values will be performed during feature selection and model preparation, depending on the requirements of the chosen machine learning model.

***

## Check for duplicate records

Duplicate records can introduce bias into exploratory analysis and machine learning models by giving additional weight to repeated observations. The dataset is checked for duplicate rows before further cleaning.

In [51]:
df.duplicated().sum()

np.int64(1313)

### Investigate duplicate records

The dataset contains duplicate records. Before removing them, the duplicate rows are inspected to determine whether they are true duplicates or repeated observations that should be retained.

In [52]:
duplicates = df[df.duplicated()]

duplicates.head(20)

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Shooting accuracy %,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks
772,Chuba Akpom,Forward,0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
778,Allan,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
785,Marco Amelia,Goalkeeper,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
790,Angeliño,Defender,0,0.0,0.0,0.0,0%,0.0,0.0,0.0,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
801,Christian Atsu,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
806,Daniel Bachmann,Goalkeeper,0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
811,Mario Balotelli,Forward,0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
815,Brandon Barker,Forward,0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
828,Essaïd Belkalem,Defender,0,0.0,0.0,0.0,0%,0.0,0.0,0.0,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
831,Ismael Bennacer,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
duplicates.nunique()

Name                      957
Position                    4
Appearances                 4
Clean sheets                2
Goals conceded              3
Tackles                     4
Tackle success %            4
Last man tackles            1
Blocked shots               3
Interceptions               4
Clearances                  4
Headed Clearance            4
Clearances off line         2
Recoveries                  4
Duels won                   4
Duels lost                  4
Successful 50/50s           3
Aerial battles won          4
Aerial battles lost         4
Own goals                   2
Errors leading to goal      2
Assists                     2
Passes                      4
Passes per match            4
Big chances created         3
Crosses                     4
Cross accuracy %            3
Through balls               3
Accurate long balls         4
Yellow cards                3
Red cards                   1
Fouls                       4
Offsides                    3
Goals     

In [57]:
df[df.duplicated(keep=False)].sort_values("Name")

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Shooting accuracy %,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks
7389,Aaron Connolly,Forward,0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6316,Aaron Connolly,Forward,0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1749,Aaron Connolly,Forward,0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2549,Aaron Connolly,Forward,0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4802,Aaron Mooy,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8200,Zidane Iqbal,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7183,Zidane Iqbal,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6088,Zidane Iqbal,Midfielder,0,NaN,NaN,0.0,0%,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7238,Ângelo,Forward,0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,...,0%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Investigation findings

The initial duplicate check identified **1,313** duplicate records. Further investigation suggested these may not be true duplicates. The merged dataset does not currently contain a `Season` column, meaning players with identical statistics across different seasons become indistinguishable. Before removing any records, the dataset construction process will be updated to preserve season information and the duplicate investigation repeated.

---